# Example: Using the Digital Twin Policy Learning Interface

This notebook illustrates how to use the generic policy learning interface in
`digital_twin_policy_learning.py` with the included facsimile dataset.

The workflow includes:

1. loading the example data
2. preparing the trajectory dataset
3. initializing the learner
4. training or loading the sequence model
5. training tabular Q-learning
6. evaluating example policies

In [ ]:
import pandas as pd
import numpy as np

from digital_twin_policy_learning import (
    TrajectoryDataset,
    MicrosimQLearner
)

## Load facsimile data
This dataset should follow the same structure as the paper’s EHR data
(id, time, action, covariates, outcomes, etc.)

In [ ]:
# Replace with your facsimile dataset (same structure as paper data)
df = pd.read_csv("your_facsimile_data.csv")

df.head()

## Define column roles
For the paper-style facsimile COVID example, use the following role assignments. Otherwise, you can use the generic version in below code cell:

```python
INTERVAL_LENGTH = 27
RNN_HIDDEN_SIZE = 128
RNN_NUM_LAYERS = 2
RNN_DROPOUT = 0.2
RNN_EPOCHS = 2000
RNN_LR = 1e-4

RNN_COVARIATE_COLS = [
    "action",
    "Age.FirstDose",
    "imm_baseline",
    "numVax",
    "gender",
    "African American",
    "Other",
    "v5-9",
    "v10-19",
    "v20-49",
    "v50",
    "c1-2",
    "c3-4",
    "c5",
    "delta",
    "omicron",
]

RNN_OUTCOME_COLS = ["sev_inf_next", "inf_next"]
RL_STATE_COLS = ["age_cat", "imm_baseline", "months_since_vax_cat"]
TIME_SINCE_ACTION_BINS = [0, 5, 7, 1000001]

patient_id_col = "id"
time_col = "month_index"
action_col = "action"

rnn_covariate_cols = RNN_COVARIATE_COLS
rnn_outcome_cols = RNN_OUTCOME_COLS
rl_state_cols = RL_STATE_COLS
```

In [ ]:
patient_id_col = "id"
time_col = "month_index"
action_col = "action"

# RNN inputs
rnn_covariate_cols = [
    # your covariates (must include action_col)
    "action",
    "Age",
    "Gender",
    "Visits",
    "imm_baseline",
    # ...
]

# RNN outputs
rnn_outcome_cols = [
    "sev_inf_next",
    "inf_next"
]

# RL state (discretized)
rl_state_cols = [
    "age_cat",
    "imm_baseline",
    "months_since_vax_cat"
]

## Build dataset

In [ ]:
dataset = TrajectoryDataset.from_long_format(
    df=df,
    patient_id_col=patient_id_col,
    time_col=time_col,
    action_col=action_col,
    rnn_covariate_cols=rnn_covariate_cols,
    rnn_outcome_cols=rnn_outcome_cols,
    rl_state_cols=rl_state_cols,

    # optional (user-defined)
    reward_outcome_col="sev_inf_next"
)

dataset.summary()

## Define optional custom functions

In [ ]:
# ===== USER-DEFINED COMPONENTS =====

def reward_fn(context):
    risk = np.asarray(context["predicted_outcomes"], dtype=float)
    reward_outcome_idx = int(context["reward_outcome_idx"])
    return -float(risk[reward_outcome_idx])

## Initialize learner

In [ ]:
learner = MicrosimQLearner(
    dataset=dataset,
    reward_fn=reward_fn,
)

## Train OR load RNN
Use one of the two options below.

For a paper-style COVID illustration, common sequence-model settings are hidden size 128, 2 layers, 2000 epochs, and learning rate `1e-4`.

In [ ]:
learner.fit_sequence_model(
    hidden_size=128,
    num_layers=2,
    epochs=2000,
    lr=1e-4
)

In [ ]:
learner.load_sequence_model(
    model_path="your_rnn_weights.pth"
)

## Train Q-learning
For a paper-style COVID illustration, common settings are `repeats_train_eval=30` and `gamma=0.99`.

In [ ]:
q_result = learner.fit_tabular_q_learning(
    repeats_train_eval=30,
    gamma=0.99
)

## Evaluate policies

In [ ]:
learned = learner.evaluate_policy("learned", epochs=5)
observed = learner.evaluate_policy("observed", epochs=5)
always = learner.evaluate_policy("all", epochs=5)
never = learner.evaluate_policy("none", epochs=5)

## Compare results

In [ ]:
print("learned:", np.mean(learned))
print("observed:", np.mean(observed))
print("always:", np.mean(always))
print("never:", np.mean(never))